# Rastros del 36 · Fase 4
## Modelo de datos relacional

En esta fase unificamos los tres datasets en una base de datos
relacional SQLite que permite consultas cruzadas entre represión y exilio.

### Tablas del modelo
- **personas**: registro unificado de todas las personas
- **eventos**: qué le pasó a cada persona (represión, exilio, combate)
- **lugares**: provincias y municipios normalizados
- **fuentes**: origen documental de cada registro

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")
print(f"📦 SQLite versión: {sqlite3.sqlite_version}")

✅ Librerías cargadas correctamente
📦 SQLite versión: 3.51.0


In [2]:
# Cargamos los tres datasets
df_euskadi = pd.read_excel('victimas_guerra_civil.xlsx')
df_andalucia = pd.read_csv('mapa_fosas_victimas_guerra_civil_andalucia.csv', 
                            sep=None, engine='python', encoding='utf-8')
df_stanbrook = pd.read_csv('stanbrook_pasajeros_limpio.csv', encoding='utf-8-sig')

print(f"✅ Euskadi:    {len(df_euskadi):>6,} registros")
print(f"✅ Andalucía:  {len(df_andalucia):>6,} registros")
print(f"✅ Stanbrook:  {len(df_stanbrook):>6,} registros")
print(f"\n📊 Total registros: {len(df_euskadi) + len(df_andalucia) + len(df_stanbrook):,}")

✅ Euskadi:    21,367 registros
✅ Andalucía:     615 registros
✅ Stanbrook:   2,614 registros

📊 Total registros: 24,596


In [3]:
import unicodedata

def normalizar(texto):
    if pd.isna(texto):
        return None
    texto = str(texto).strip()
    return texto

# Tabla personas desde Euskadi
personas_euskadi = pd.DataFrame({
    'id_persona': ['EUS_' + str(i) for i in range(len(df_euskadi))],
    'nombre': df_euskadi['Nombre'].apply(normalizar),
    'apellidos': df_euskadi['Apellidos'].apply(normalizar),
    'edad': pd.to_numeric(df_euskadi['Edad'], errors='coerce'),
    'profesion': df_euskadi['Profesión'].apply(normalizar),
    'filiacion': df_euskadi['Filiación política/social'].apply(normalizar),
    'provincia_nacimiento': df_euskadi['Provincia nacimiento'].apply(normalizar),
    'provincia_fallecimiento': df_euskadi['Provincia fallecimiento'].apply(normalizar),
    'causa_muerte': df_euskadi['Causa muerte'].apply(normalizar),
    'tipo_registro': 'victima',
    'comunidad': 'País Vasco',
    'fuente': 'Open Data Euskadi'
})

# Tabla personas desde Stanbrook
personas_stanbrook = pd.DataFrame({
    'id_persona': ['STB_' + str(i) for i in range(len(df_stanbrook))],
    'nombre': df_stanbrook['Nombre'].apply(normalizar),
    'apellidos': (df_stanbrook['Apellido1'] + ' ' + df_stanbrook['Apellido2']).apply(normalizar),
    'edad': pd.to_numeric(df_stanbrook['Edad'], errors='coerce'),
    'profesion': df_stanbrook['Profesion_es'].apply(normalizar),
    'filiacion': None,
    'provincia_nacimiento': None,
    'provincia_fallecimiento': None,
    'causa_muerte': None,
    'tipo_registro': 'exiliado',
    'comunidad': 'Varios',
    'fuente': 'Fundación Pablo Iglesias — Stanbrook'
})

# Unificamos en una sola tabla
df_personas = pd.concat([personas_euskadi, personas_stanbrook], ignore_index=True)

print(f"✅ Tabla personas construida: {len(df_personas):,} registros")
print(f"\n--- DISTRIBUCIÓN POR TIPO ---")
print(df_personas['tipo_registro'].value_counts())
print(f"\n--- COLUMNAS ---")
print(list(df_personas.columns))

✅ Tabla personas construida: 23,981 registros

--- DISTRIBUCIÓN POR TIPO ---
tipo_registro
victima     21367
exiliado     2614
Name: count, dtype: int64

--- COLUMNAS ---
['id_persona', 'nombre', 'apellidos', 'edad', 'profesion', 'filiacion', 'provincia_nacimiento', 'provincia_fallecimiento', 'causa_muerte', 'tipo_registro', 'comunidad', 'fuente']


In [4]:
# Tabla lugares — normalizamos todas las provincias que aparecen
coordenadas = {
    'Araba/Álava': (42.8469, -2.6727),
    'Bizkaia': (43.2630, -2.9350),
    'Gipuzkoa': (43.3128, -1.9754),
    'Asturias': (43.3614, -5.8593),
    'Cantabria': (43.1828, -3.9878),
    'Burgos': (42.3440, -3.6969),
    'Madrid': (40.4168, -3.7038),
    'Navarra': (42.6954, -1.6761),
    'Teruel': (40.3456, -1.1065),
    'Lleida': (41.6176, 0.6200),
    'Tarragona': (41.1189, 1.2445),
    'Barcelona': (41.3851, 2.1734),
    'Zaragoza': (41.6561, -0.8773),
    'Rioja (La)': (42.2871, -2.5396),
    'Sevilla': (37.3891, -5.9845),
    'Huelva': (37.2614, -6.9447),
    'Cádiz': (36.5271, -6.2886),
    'Granada': (37.1773, -3.5986),
    'Málaga': (36.7213, -4.4213),
    'Córdoba': (37.8882, -4.7794),
    'Jaén': (37.7796, -3.7849),
    'Almería': (36.8340, -2.4637),
    'Valencia/València': (39.4699, -0.3763),
    'Alicante/Alacant': (38.3452, -0.4810),
    'Castellón/Castelló': (39.9864, -0.0513),
    'Murcia': (37.9922, -1.1307),
}

lugares = []
id_lugar = 1
for provincia, (lat, lon) in coordenadas.items():
    lugares.append({
        'id_lugar': id_lugar,
        'provincia': provincia,
        'latitud': lat,
        'longitud': lon,
        'pais': 'España'
    })
    id_lugar += 1

df_lugares = pd.DataFrame(lugares)
print(f"✅ Tabla lugares construida: {len(df_lugares)} provincias")
print(df_lugares.head(5).to_string())

✅ Tabla lugares construida: 26 provincias
   id_lugar    provincia  latitud  longitud    pais
0         1  Araba/Álava  42.8469   -2.6727  España
1         2      Bizkaia  43.2630   -2.9350  España
2         3     Gipuzkoa  43.3128   -1.9754  España
3         4     Asturias  43.3614   -5.8593  España
4         5    Cantabria  43.1828   -3.9878  España


In [5]:
# Creamos la base de datos
conn = sqlite3.connect('rastros_del_36.db')

# Guardamos las tablas
df_personas.to_sql('personas', conn, if_exists='replace', index=False)
df_lugares.to_sql('lugares', conn, if_exists='replace', index=False)

# Tabla fosas de Andalucía
df_andalucia_clean = df_andalucia.copy()
if 'Unnamed: 0' in df_andalucia_clean.columns:
    df_andalucia_clean = df_andalucia_clean.drop(columns=['Unnamed: 0'])
df_andalucia_clean.to_sql('fosas', conn, if_exists='replace', index=False)

print("✅ Base de datos creada: rastros_del_36.db")
print("\n--- TABLAS EN LA BASE DE DATOS ---")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tablas = cursor.fetchall()
for tabla in tablas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla[0]}")
    count = cursor.fetchone()[0]
    print(f"  📋 {tabla[0]:20} → {count:,} registros")

✅ Base de datos creada: rastros_del_36.db

--- TABLAS EN LA BASE DE DATOS ---
  📋 personas             → 23,981 registros
  📋 lugares              → 26 registros
  📋 fosas                → 615 registros


In [6]:
# Consulta 1 — ¿Cuántas víctimas y exiliados por tipo?
print("=== CONSULTA 1: Distribución por tipo de registro ===")
resultado = pd.read_sql_query("""
    SELECT tipo_registro, COUNT(*) as total
    FROM personas
    GROUP BY tipo_registro
""", conn)
print(resultado.to_string(index=False))

# Consulta 2 — Profesiones más frecuentes en víctimas vs exiliados
print("\n=== CONSULTA 2: Top 5 profesiones por tipo ===")
resultado2 = pd.read_sql_query("""
    SELECT tipo_registro, profesion, COUNT(*) as total
    FROM personas
    WHERE profesion IS NOT NULL
    GROUP BY tipo_registro, profesion
    ORDER BY tipo_registro, total DESC
    LIMIT 20
""", conn)
print(resultado2.to_string(index=False))

# Consulta 3 — Edad media por tipo
print("\n=== CONSULTA 3: Edad media por tipo de registro ===")
resultado3 = pd.read_sql_query("""
    SELECT tipo_registro,
           ROUND(AVG(edad), 1) as edad_media,
           MIN(edad) as edad_minima,
           MAX(edad) as edad_maxima,
           COUNT(*) as total_con_edad
    FROM personas
    WHERE edad IS NOT NULL AND edad > 0 AND edad < 100
    GROUP BY tipo_registro
""", conn)
print(resultado3.to_string(index=False))

=== CONSULTA 1: Distribución por tipo de registro ===
tipo_registro  total
     exiliado   2614
      victima  21367

=== CONSULTA 2: Top 5 profesiones por tipo ===
tipo_registro     profesion  total
     exiliado    Agricultor    215
     exiliado Sin profesión    199
     exiliado      Mecánico    142
     exiliado        Chófer    115
     exiliado   Ferroviario    100
     exiliado      Empleado     82
     exiliado       Albañil     80
     exiliado      Contable     61
     exiliado   Metalúrgico     54
     exiliado      Zapatero     52
     exiliado     Jornalero     48
     exiliado   Comerciante     46
     exiliado      Panadero     40
     exiliado    Carpintero     40
     exiliado    Estudiante     36
     exiliado      Ajusteur     35
     exiliado       Maestro     33
     exiliado     Peluquero     30
     exiliado    Industriel     30
     exiliado  Electricista     30

=== CONSULTA 3: Edad media por tipo de registro ===
tipo_registro  edad_media  edad_minima  edad_ma

In [7]:
# Consulta 4 — Provincias con más víctimas unidas a sus coordenadas
print("=== CONSULTA 4: Víctimas por provincia con coordenadas ===")
resultado4 = pd.read_sql_query("""
    SELECT p.provincia_fallecimiento as provincia,
           COUNT(*) as total_victimas,
           l.latitud,
           l.longitud
    FROM personas p
    LEFT JOIN lugares l ON p.provincia_fallecimiento = l.provincia
    WHERE p.tipo_registro = 'victima'
    AND p.provincia_fallecimiento IS NOT NULL
    GROUP BY p.provincia_fallecimiento
    ORDER BY total_victimas DESC
    LIMIT 10
""", conn)
print(resultado4.to_string(index=False))

# Consulta 5 — Jóvenes menores de 18 en el exilio
print("\n=== CONSULTA 5: Menores de 18 en el Stanbrook ===")
resultado5 = pd.read_sql_query("""
    SELECT nombre, apellidos, edad, profesion
    FROM personas
    WHERE tipo_registro = 'exiliado'
    AND edad < 18
    AND edad > 0
    ORDER BY edad ASC
    LIMIT 10
""", conn)
print(resultado5.to_string(index=False))

# Consulta 6 — Perfil más común de víctima ejecutada
print("\n=== CONSULTA 6: Perfil de ejecutados ===")
resultado6 = pd.read_sql_query("""
    SELECT profesion, filiacion, COUNT(*) as total
    FROM personas
    WHERE tipo_registro = 'victima'
    AND causa_muerte LIKE '%ejecutad%'
    AND profesion IS NOT NULL
    GROUP BY profesion, filiacion
    ORDER BY total DESC
    LIMIT 10
""", conn)
print(resultado6.to_string(index=False))

=== CONSULTA 4: Víctimas por provincia con coordenadas ===
              provincia  total_victimas  latitud  longitud
                Bizkaia            9655  43.2630   -2.9350
               Gipuzkoa            3237  43.3128   -1.9754
            Araba/Álava            2254  42.8469   -2.6727
Ezezaguna / Desconocida            1481      NaN       NaN
               Asturias             666  43.3614   -5.8593
              Cantabria             609  43.1828   -3.9878
                 Teruel             426  40.3456   -1.1065
                 Madrid             300  40.4168   -3.7038
                 Lleida             248  41.6176    0.6200
              Tarragona             187  41.1189    1.2445

=== CONSULTA 5: Menores de 18 en el Stanbrook ===
   nombre           apellidos  edad profesion
     José   Ballestero Bayona   1.0       NaN
    Saray        Bolano Paulo   1.0       NaN
María Luz Caballero Caballero   1.0       NaN
  Eugenia        Estefin Dura   1.0   Roumain
 Libertad  

### El poder del modelo relacional

Con una base de datos relacional podemos hacer preguntas que 
antes requerían cruzar manualmente tres archivos distintos:

- ¿Qué profesiones tenían los ejecutados?
- ¿Qué provincias concentraron más muertes y dónde están?
- ¿Qué menores viajaron en el Stanbrook?

SQL nos permite formular estas preguntas en segundos sobre
casi 24.000 registros unificados.

Los nombres de los bebés del Stanbrook — Libertad, Iberia — 
son en sí mismos un documento histórico: la República
como proyecto de vida, incluso en la huida.

In [8]:
# Cerramos la conexión
conn.close()
print("✅ Base de datos guardada: rastros_del_36.db")
print("📦 Archivo listo para subir a GitHub")

# Verificamos el tamaño
import os
tamaño = os.path.getsize('rastros_del_36.db') / (1024*1024)
print(f"💾 Tamaño: {tamaño:.2f} MB")

✅ Base de datos guardada: rastros_del_36.db
📦 Archivo listo para subir a GitHub
💾 Tamaño: 6.05 MB
